[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/longas14/Fisica_Computacional_1/blob/main/notebooks/pandas.ipynb)

---
# Física Computacional 1
Robinson Longas Bedoya  
*Facultad de Ciencias Exactas y Aplicadas - Institución Universitaria ITM*  
[robinsonlongas@itm.edu.co](mailto:robinsonlongas@itm.edu.co) | [robinsonlongas3586@correo.itm.edu.co](mailto:robinsonlongas3586@correo.itm.edu.co)

---

### Temas principales:
* **Introducción a pandas: series y data frames**
#### 📚 Bibliografía recomendada
*  Documentación oficial: [`Pandas`](https://pandas.pydata.org/docs/)
*  Documentación no oficial: [`Ejemplos`](https://aaltoscicomp.github.io/python-for-scicomp/pandas/)
*  Documentación no oficial: [`Ejemplos`](https://joserzapata.github.io/courses/python-ciencia-datos/pandas/)
* *Python Data science handbook: essential tools for working with data*, Jake VanderPlas.

# <span style="color:red; font-style:italic;"> Breve introducción a Pandas</span>
 
---
Pandas es una herramienta de manipulación de datos ampliamente utilizada en el análisis de datos.

En computación científica, Python estándar (`list`, `dict`) presenta alguinas limitaciones de rendimiento:
* Las listas de Python almacenan **punteros a objetos dispersos en memoria**, generando sobrecarga de punteros (*pointer indirection*) y perdiendo la localidad espacial de la memoria caché de la CPU.
* **NumPy** resuelve esto con el `ndarray`: bloques contiguos de memoria de tipo homogéneo en C/Fortran con soporte para operaciones vectorizadas sin bucles de Python.

### La limitación de NumPy frente a datos tabulares
NumPy exige homogeneidad estricta. Cuando se trabaja con escaneos de física que mezclan números de punto flotante en escala logarítmica, identificadores enteros, banderas booleanas o metadatos, estructurar y consultar los datos se vuelve propenso a errores de indexación.

### El rol de Pandas
Pandas extiende NumPy combinando la velocidad de bajo nivel con la semántica de **bases de datos relacionales (SQL)**:
* **Alineación implícita:** Las operaciones entre estructuras se alinean automáticamente mediante índices.
* **Manejo de metadatos:** Cada columna conserva su tipo de dato nativo (`dtype`) y su etiqueta conceptual.
* **Motor interno:** Agrupa internamente columnas del mismo tipo contiguas en memoria (*BlockManager*) para garantizar que las transformaciones vectorizadas no pierdan la velocidad de C.

In [ ]:
# librerías básicas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# configuración de estilo
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Seleccionar los stilos 
plt.style.use('seaborn-v0_8-ticks') 

# Configuración de LaTeX
plt.rcParams.update({"text.usetex": True, "text.latex.preamble": r"\usepackage{bm} \usepackage{amsmath}", "font.family": "serif","font.serif": ["Computer Modern Roman"]})

#Configuración de Ticks
plt.rcParams['xtick.major.size'] = 10
plt.rcParams['xtick.minor.size'] = 6
plt.rcParams['ytick.major.size'] = 10
plt.rcParams['ytick.minor.size'] = 6

# Forzar que las marcas apunten hacia ADENTRO (estilo paper)
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.right'] = True

In [ ]:
# cargar datos
results = pd.read_csv("resultados_C.csv") 
print("==" * 30)

# númerop de filas  columnas
print(f"Tipo de objeto tabla   : {type(results)}")
print(f"Dimensiones DataFrame  : {results.shape} (filas, columnas)")
print(f"Columnas detectadas    : {list(results.columns)}")
print("==" * 30)

# Información
print("Información estructural y memoria:")
results.info()
print("==" * 30)

# Ver las primeras 5 filas 
display(results.head()) 
print("==" * 30)
print("==" * 30)

#Extraer una columna aislada (Series 1D)
omega_series = results["Omega"]
print(f"Tipo de objeto columna : {type(omega_series)}")
print(f"Dimensiones de Series  : {omega_series.shape}")

## Cargar datos: `pd.read_csv`

Al invocar `pd.read_csv`, Pandas hace lo siguiente:
1. Lee el flujo de bytes del archivo a bajo nivel.
2. Evalúa los valores de cada columna para definir si son `float64`, `int64`, `str`, etc.
3. Genera un `RangeIndex(0, N-1)` en memoria de consumo $O(1)$.

### Inspección Inicial de las Variables Físicas

El dataset contiene las siguientes variables físicas:
* `mN1`: Masa del singlete fermiónico $M_{N_1}$ [GeV].
* `MV`: Masa del mediador vectorial $M_V$ [GeV].
* `TRH`: Temperatura de recalentamiento $T_{\text{RH}}$ [GeV].
* `Omega`: Densidad reliquia calculada $\Omega h^2$.
* `Gamma_over_H`: Razón $\Gamma/H$, que determina si el proceso estuvo en equilibrio ($\Gamma/H \gtrsim 1$) o si operó como freeze-in ($\Gamma/H \ll 1$).
* `yX11`: Acoplamiento Yukawa relevante para la interacción.

---

## Estructuras fundamentales
Para entender cómo opera Pandas sobre los datos de un escaneo numérico, es indispensable distinguir sus dos estructuras centrales y cómo se mapean en la memoria RAM:

* **`pd.Series` (1D):** Arreglo unidimensional etiquetado. Es un `np.ndarray` contiguo acoplado a un objeto `Index` que mapea etiquetas a posiciones de memoria.
* **`pd.DataFrame` (2D):** Diccionario ordenado de Series que comparten el mismo `Index` horizontal y vertical.

```text
                  DataFrame (2D)
    ┌──────────────────────────────────────────────┐
    │           Columnas (pd.Index de ejes)        │
    │    mN1       MV       TRH       Omega   ...  │
────┼──────────────────────────────────────────────┤
 I  │   0.001    200.0    0.01      0.1198   ...   │ ──> Fila (registro físico individual)
 n  │   0.001    200.0    0.1       0.1205   ...   │
 d  │    ...      ...      ...        ...          │
 e  │                                              │
 x  │    └── Cada columna individual es una ───────┘
    │                 Series (1D)                  │

---

## Limpieza de los datos

En simulaciones numéricas, las rutinas de integración de ecuaciones diferenciales (como la ecuación de Boltzmann para $\Omega h^2$) pueden divergir, producir de *overflow* o no converger en regiones asintóticas del espacio de parámetros.

Antes de cualquier análisis aplicamos un protocolo de limpieza en cuatro etapas:

1. **Homogeneización de infinitos (`replace`):**  
   Convertimos `+inf` y `-inf` en `np.nan` para tratarlos de manera uniforme con los métodos nativos de valores faltantes de Pandas.
2. **Eliminación de registros incompletos (`dropna`):**  
   Descartamos filas donde las variables físicas esenciales (por ejemplo `mN1` o `Omega`) no hayan convergido (`NaN`).
3. **Filtros físicos. (En este caso `data['Omega'] > 0`):**  
   La densidad reliquia $\Omega h^2$ debe ser estrictamente positiva. Este paso garantiza que las transformaciones a escala logarítmica ($\log_{10}\Omega$) estén bien definidas.
4. **Reconstrucción del índice (`reset_index(drop=True)`):**  
   Al descartar filas, el índice original conserva huecos (`[0, 1, 4, 7, ...]`). El reseteo restaura una secuencia continua de enteros `[0, 1, 2, ...]`, evitando desalineaciones lógicas en bucles o accesos posicionales.

In [ ]:
data = results.replace([np.inf, -np.inf], np.nan) # Reemplazar infinitos por NaN 
data = data.dropna(subset=['mN1', 'Omega']) # Eliminamos filas donde mN1 o Omega son NaN
data = data[data['Omega']>0] # elimina las filas donde omega es negativo o cero
data = data.reset_index(drop=True) #Resetear el índice para que las filas vuelvan a enumerarse desde 0

print(f"Puntos que contiende el archivo de entrada: {len(results)}")
print(f"Puntos que sobreviven a la limpieza       : {len(data)}")
print(f"Puntos descartados                        : {len(results) - len(data)}")

---

## Manipulación del `DataFrame`

Pdemos crear nuevas columnas 

1. **Razón entre masas ($n = M_V / m_{N_1}$):**
   Determina el espacio de fase disponible y el canal cinemático dominante:
   * $n > 2$: El decaimiento del mediador en pares de singletes ($V \to N_1 N_1$) está cinemáticamente abierto.
   * $n \le 2$: El canal de decaimiento está bloqueado, forzando la producción por dispersión $2 \to 2$ o aniquilación.

2. **Clasificación del Régimen de Materia Oscura (`DMRegime`):**
   La tasa de interacción comparada con la tasa de expansión de Hubble ($\Gamma / H$) gobierna el mecanismo térmico:
   * $\Gamma / H < 1$: Las partículas jamás alcanzan el equilibrio térmico con el baño del Modelo Estándar (**FIMP** / *Freeze-in*).
   * $\Gamma / H \ge 1$: Las interacciones entran en equilibrio térmico antes de desacoplarse (**WIMP** / *Freeze-out*).

Para evitar bucles `for`, utilizamos `np.where(condición, valor_si_true, valor_si_false)`, el cual opera a nivel de bloques continuos de memoria C.

> **Detalle técnico sobre `.copy()`:** Al segmentar `datafimp` y `datawimp`, invocar `.copy()` obliga a Pandas a reservar un nuevo búfer de memoria contiguo. Si se omite, Pandas genera una *vista* (*view*), lo que provocaría advertencias de `SettingWithCopyWarning` al intentar añadir nuevas columnas o modificar estos subconjuntos más adelante.

In [ ]:
# definición de n
data["n"] = data["MV"] / data["mN1"]

# definición del régimen FIMP or WIMP
data["DMRegime"] = np.where(data["Gamma_over_H"] < 1.0, "FIMP", "WIMP")

# Diagnóstico de frecuencias con value_counts()
print("Puntos por régimen:")
print(data["DMRegime"].value_counts())
print("==" * 30)

# Partimos el dataset mediante máscaras booleanas
datafimp = data[data["DMRegime"] == "FIMP"].copy()
datawimp = data[data["DMRegime"] == "WIMP"].copy()

print(f"Total puntos : {len(data)}")
print(f"Puntos FIMP   : {len(datafimp)} ({len(datafimp)/len(data):.1%})")
print(f"Puntos WIMP   : {len(datawimp)} ({len(datawimp)/len(data):.1%})")

In [ ]:
# Valores de TRH explorados 
valores_trh = sorted(datafimp["TRH"].unique())
print(f"Valores de TRH presentes en FIMP: {valores_trh}")
# Valores de MV explorados 
valores_MV = sorted(datafimp["MV"].unique())
print(f"Valores de MV presentes en FIMP: {valores_MV}")
# Valores de yX11 explorados 
valores_Y = sorted(datafimp["yX11"].unique())
print(f"Valores de yX11 presentes en FIMP: {valores_Y}")

---

## Estudio de Caso: Dependencia Térmica de $\Omega_N h^2$ para Cortes Fijos de $M_V$

Para ilustrar de forma tangible por qué `groupby` es una herramienta central en análisis de datos, abordaremos un problema típico de física computacional en dos etapas:
1. **Enfoque Explícito ("A mano"):** Construyendo máscaras booleanas y filtrando subconjuntos paso a paso.
2. **Refactorización con `groupby`:** Reemplazando la sobrecarga de código manual con una partición vectorizada y estructurada.

---

### Enfoque 1: Filtrado Explícito y Construcción Manual

Supongamos que deseamos fijar el acoplamiento Yukawa en $y_{X_{11}} = 10^{-13}$ y analizar el comportamiento de la densidad reliquia $\Omega_N h^2$ frente a la masa del singlete $m_N$ para tres masas del mediador vectorial:
$$M_V \in \{200, 500, 800\}\ \text{GeV}$$
superponiendo diferentes curvas de temperatura de recalentamiento cosmológico $T_{\text{RH}}$.

De forma manual, el procedimiento exige:
* Construir un sub-DataFrame independiente por cada masa (`m20013`, `m50013`, `m80013`).
* En cada paso de un bucle sobre $T_{\text{RH}}$, evaluar manualmente máscaras booleanas con `np.isclose()`.
* Replicar manualmente las instrucciones de formateo de ejes para `ax1`, `ax2` y `ax3`.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15,5))

configs = [
    (1e-2, 'navy',  r'$T_{\text{RH}} = 0.01 \text{ GeV}$'),
    (1e-1, 'green', r'$T_{\text{RH}} = 0.1 \text{ GeV}$'),
    (1e0,  'blue',  r'$T_{\text{RH}} = 1 \text{ GeV}$'),
    (1e1,  'cyan',  r'$T_{\text{RH}} = 10 \text{ GeV}$'),
    (1e2,  'red',   r'$T_{\text{RH}} = 100 \text{ GeV}$'),
    (1e4,  'black', r'$T_{\text{RH}} = 10 \text{ TeV}$'),
]

#Filtrado explícito a mano por cada corte de masa MV y para yX11 = 1e-13
m20013 = datafimp[(datafimp.MV==200) & (datafimp.yX11==1e-13) ]
m50013 = datafimp[(datafimp.MV==500) & (datafimp.yX11==1e-13) ]
m80013 = datafimp[(datafimp.MV==800) & (datafimp.yX11==1e-13) ]

for trh_val, color, label in configs:
    mask2 = np.isclose(m20013['TRH'], trh_val) # máscara para cada valor de TRH
    mask5 = np.isclose(m50013['TRH'], trh_val)
    mask8 = np.isclose(m80013['TRH'], trh_val)

    sub2 = m20013[mask2].sort_values('mN1') # se aplica la máscara y se organiza por valores de mN1
    sub5 = m50013[mask5].sort_values('mN1')
    sub8 = m80013[mask8].sort_values('mN1')
    
    ax1.plot( sub2['mN1'],  sub2['Omega'],  color=color,  linestyle='-',   lw=2,   label=label )
    ax2.plot( sub5['mN1'],  sub5['Omega'],  color=color,  linestyle='-',   lw=2,   label=label )
    ax3.plot( sub8['mN1'],  sub8['Omega'],  color=color,  linestyle='-',   lw=2,   label=label )

# línea experminetal de Planck
omega_central = 0.12
sigma = 0.003

# plots 
ax1.set_yscale('log')
ax1.set_xscale('log')
ax1.set_xlim(1e-3, 1e3)
ax1.set_ylim(1e-60, 1e20)
ax1.set_xlabel(r'$\boldsymbol{m_N\,\rm[GeV]}$', size=20)
ax1.set_ylabel(r'$\boldsymbol{\Omega_{N} h^2}$', size=20)
ax1.set_title(r'$\boldsymbol{m_V = 200 \,\rm[GeV]\;; y = 10^{-13}}$', size=13)
ax1.axhspan(omega_central - 3*sigma, omega_central + 3*sigma, color='gray', alpha=0.5)
ax1.legend(prop={'size': 9}, loc='upper left', frameon=True, fancybox=True, framealpha=0.5)

ax2.set_yscale('log')
ax2.set_xscale('log')
ax2.set_xlim(1e-3, 1e3)
ax2.set_ylim(1e-60, 1e20)
ax2.set_xlabel(r'$\boldsymbol{m_N\,\rm[GeV]}$', size=20)
ax2.set_title(r'$\boldsymbol{m_V = 500 \,\rm[GeV]\;; y = 10^{-13}}$', size=13)
ax2.axhspan(omega_central - 3*sigma, omega_central + 3*sigma, color='gray', alpha=0.5)
ax2.legend(prop={'size': 9}, loc='upper left', frameon=True, fancybox=True, framealpha=0.5)

ax3.set_yscale('log')
ax3.set_xscale('log')
ax3.set_xlim(1e-3, 1e3)
ax3.set_ylim(1e-60, 1e20)
ax3.set_xlabel(r'$\boldsymbol{m_N\,\rm[GeV]}$', size=20)
ax3.set_title(r'$\boldsymbol{m_V = 800 \,\rm[GeV]\;; y = 10^{-13}}$', size=13)
ax3.axhspan(omega_central - 3*sigma, omega_central + 3*sigma, color='gray', alpha=0.5)
ax3.legend(prop={'size': 9}, loc='upper left', frameon=True, fancybox=True, framealpha=0.5)


---

### Limitaciones del Enfoque Manual

1. **Rigidez y Deuda Técnica:** Si queremos evaluar 6 valores de $M_V$ en lugar de 3, tendríamos que declarar `m30013`, `m40013`, duplicar `maskN`, y copiar 20 líneas adicionales de código por cada panel.
2. **Operaciones O(N) Repetitivas:** En cada iteración del bucle `for`, Pandas recorre toda la columna evaluando `np.isclose()`, en lugar de indexar de golpe la partición de datos.
3. **Mantenibilidad:** Cambiar un límite de graficación (`xlim`, tamaño de letra) obliga a editar 3 bloques de código independientes.

---

### Enfoque 2: Automatización Idiomática con `groupby`

`groupby` resuelve este problema delegando la partición de datos al motor interno de indexación de Pandas:

1. **Un solo filtro global:** Seleccionamos únicamente los registros con $y_{X_{11}} = 10^{-13}$ y las masas de interés usando `.isin([200, 500, 800])`.
2. **Mapeo Eje-Grupo con Diccionarios:** Asociamos cada eje `ax` a su respectivo $M_V$ mediante `dict(zip(masas_mv, axes))`.
3. **Partición Automática con `groupby(['MV', 'TRH'])`:** Pandas segmenta el dataset en subtablas indexadas directamente por el par $(M_V, T_{\text{RH}})$, permitiendo iterar sin máscaras manuales.
4. **Bucle de estilo unificado:** Las propiedades de visualización y límites de escala se aplican en un único bucle compacto sobre los ejes.

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Filtro base vectorial
masas_mv = [200, 500, 800]
yuka = 1e-13
sub_data = datafimp[ (datafimp['MV'].isin(masas_mv)) & (datafimp['yX11']== yuka)].drop_duplicates(subset=['MV', 'TRH', 'mN1']) # elimina columnas repetidas en subset

# Diccionario de metadatos de estilo por cada escala térmica TRH
estilos_trh = {
    1e-2: ("navy", r"$T_{\text{RH}} = 0.01 \text{ GeV}$"),
    1e-1: ("green", r"$T_{\text{RH}} = 0.1 \text{ GeV}$"),
    1e0: ("blue", r"$T_{\text{RH}} = 1 \text{ GeV}$"),
    1e1: ("cyan", r"$T_{\text{RH}} = 10 \text{ GeV}$"),
    1e2: ("red", r"$T_{\text{RH}} = 100 \text{ GeV}$"),
    1e4: ("black", r"$T_{\text{RH}} = 10 \text{ TeV}$"),
}

# mapeo de paneles (sharey=True alinea la escala física de los 3 paneles)
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
ax_map = dict(zip(masas_mv, axes))

# Uso de GROUPBY
# El bucle itera sobre cada combinación única (MV, TRH) presente en los datos
for (mv_val, trh_val), grupo in sub_data.groupby(["MV", "TRH"]):
    # Localizar estilo correspondiente para la temperatura
    for trh_key, (color, label) in estilos_trh.items():
        if np.isclose(trh_val, trh_key):
            curva = grupo.sort_values("mN1")
            ax_map[mv_val].plot(
                curva["mN1"],
                curva["Omega"],
                color=color,
                linestyle="-",
                lw=2,
                label=label,
            )
            break

# Configuración unificada de los paneles
omega_central, sigma = 0.12, 0.003 # Planck

for mv_val, ax in ax_map.items():
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(1e-3, 1e3)
    ax.set_ylim(1e-60, 1e10)
    ax.set_xlabel(r"$\boldsymbol{m_N\,\rm[GeV]}$", size=16)
    ax.set_title(rf"$\boldsymbol{{m_V = {mv_val} \,\rm[GeV]\;; y = 10^{{-13}}}}$", size=13 )

    # Banda observacional de Planck (3-sigma)
    ax.axhspan(
        omega_central - 3 * sigma,
        omega_central + 3 * sigma,
        color="gray",
        alpha=0.5,
    )
    ax.legend(
        prop={"size": 8.5},
        loc="upper left",
        frameon=True,
        fancybox=True,
        framealpha=0.5,
    )

axes[0].set_ylabel(r"$\boldsymbol{\Omega_{N} h^2}$", size=16)
plt.tight_layout()
plt.show()


## 8. ¿Cómo Funciona `groupby` Detrás de los Gráficos?

Al observar los paneles anteriores, es evidente que cada curva física representa un conjunto aislado de puntos condicionado a dos variables de estado:
1. Una **masa de mediador fija** ($M_V = 200, 500 \text{ ó } 800\text{ GeV}$).
2. Una **temperatura de recalentamiento fija** ($T_{\text{RH}} = 10^{-2}, 10^{-1}, \dots, 10^4\text{ GeV}$).

Para trazar estas curvas, en el enfoque manual tuvimos que escribir máscaras booleanas redundantes (`mask2`, `mask5`, `mask8`). A continuación desglosamos la mecánica interna de `groupby` y por qué es el motor natural para generar gráficos por bloques o categorías.

---

### 1. La Clave: `groupby` como Generador de Tuplas `(Clave, Sub-DataFrame)`

Cuando ejecutamos:

```python
sub_data.groupby(['MV', 'TRH'])
for (mv_val, trh_val), grupo in sub_data.groupby(['MV', 'TRH']):
    │                  │
    │                  └─> 'grupo': Sub-DataFrame con SOLO los datos de ese par
    └────────────────────> '(mv_val, trh_val)': Tupla con las etiquetas del corte

sub_data (Filtrado global: y = 1e-13)
        ┌─────────────────────────────────────────────────────────┐
        │  MV     TRH       mN1       Omega      Gamma_over_H ... │
        │ 200    0.01      0.001     1.2e-40         ...          │
        │ 200    0.01      0.010     3.4e-38         ...          │
        │ 200    10.0      0.001     5.1e-12         ...          │
        │ 500    0.01      0.001     8.9e-45         ...          │
        └────────────────────────────┬────────────────────────────┘
                                     │
                    [ groupby(['MV', 'TRH']) ]
                                     │
         ┌───────────────────────────┼───────────────────────────┐
         ▼                           ▼                           ▼
  Iteración 1                 Iteración 2                 Iteración 3
  Clave: (200, 0.01)          Clave: (200, 10.0)          Clave: (500, 0.01)
  grupo (Sub-DF):             grupo (Sub-DF):             grupo (Sub-DF):
  ┌─────────────────┐         ┌─────────────────┐         ┌─────────────────┐
  │ mN1    Omega    │         │ mN1    Omega    │         │ mN1    Omega    │
  │ 0.001  1.2e-40  │         │ 0.001  5.1e-12  │         │ 0.001  8.9e-45  │
  │ 0.010  3.4e-38  │         │  ...     ...    │         │  ...     ...    │
  └────────┬────────┘         └────────┬────────┘         └────────┬────────┘
           │                           │                           │
           ▼                           ▼                           ▼
   ax_map[200].plot(...)       ax_map[200].plot(...)       ax_map[500].plot(...)
   (Curva azul en panel 1)     (Curva roja en panel 1)     (Curva azul en panel 2)    

---

## Agrupación y Reducción con `groupby`: Paradigma Split-Apply-Combine

El método `.groupby()` de Pandas formaliza el patrón analítico **Split-Apply-Combine** (Dividir, Aplicar y Combinar), introducido originalmente en computación estadística por Hadley Wickham.

En escaneos numéricos multidimensionales, rara vez analizamos todas las variables de forma simultánea. Habitualmente necesitamos fijar uno o dos parámetros de control (como la masa del mediador $M_V$ o del neutrino $m_{N_1}$) y evaluar el comportamiento de los observables ($\Omega h^2$, $y_{X_{11}}$, $\Gamma/H$) dentro de cada porción del espacio de fases.

---

### Mecánica Interna: Las Tres Fases

```text
                        DataFrame Original
                 ┌──────────────────────────────┐
                 │    MV     mN1    Omega  ...  │
                 │   200     1.0    0.040       │
                 │   200     1.0    0.119       │
                 │   500     1.0    0.085       │
                 │   500     2.0    0.121       │
                 └──────────────┬───────────────┘
                                │
                    [ 1. SPLIT por 'MV' ]
                                ▼
         Grupo MV = 200                    Grupo MV = 500
   ┌───────────────────────┐         ┌───────────────────────┐
   │  mN1   Omega    ...   │         │  mN1   Omega    ...   │
   │  1.0   0.040          │         │  1.0   0.085          │
   │  1.0   0.119          │         │  2.0   0.121          │
   └───────────┬───────────┘         └───────────┬───────────┘
               │                                 │
               ▼                                 ▼
    [ 2. APPLY: mean() ]              [ 2. APPLY: mean() ]
         Omega = 0.0795                    Omega = 0.1030
               │                                 │
               └────────────────┬────────────────┘
                                │
                    [ 3. COMBINE en Series/DF ]
                                ▼
                           MV    Omega
                          200   0.0795
                          500   0.1030

In [ ]:
# Agrupamos por mediador y temperatura
MVTRH = datafimp.groupby(["MV", "TRH"])
# extraemos MV = 200 GeV y TRH = 0.01 GeV, usamos una tupla:
corte2 = MVTRH.get_group((200.0, 0.01))
corte5 = MVTRH.get_group((500.0, 0.01))
corte8 = MVTRH.get_group((800.0, 0.01))
plt.plot(corte2[corte2.yX11==1e-13].mN1,corte2[corte2.yX11==1e-13].Omega, color='blue',lw=2, label=r"$\boldsymbol{M_{V} = 200 \,\rm[GeV] }$")
plt.plot(corte5[corte5.yX11==1e-13].mN1,corte5[corte5.yX11==1e-13].Omega, color ='red', lw=2, label=r"$\boldsymbol{M_{V} = 500 \,\rm[GeV] }$")
plt.plot(corte8[corte8.yX11==1e-13].mN1,corte8[corte8.yX11==1e-13].Omega, color ='green', lw=2, label=r"$\boldsymbol{M_{V} = 800 \,\rm[GeV] }$")
plt.yscale('log')
plt.xscale('log')
#plt.ylim(1e-60, 1e10)
plt.xlabel(r"$\boldsymbol{m_N\,\rm[GeV]}$", size=16)
plt.title(rf"$\boldsymbol{{T_{{RH}} = {0.01} \,\rm[GeV]\;; y = 10^{{-13}}}}$", size=13 )
plt.ylabel(r"$\boldsymbol{\Omega_{N} h^2}$", size=16)
plt.legend(loc="lower left")

---

## 8.2. Superposición de Curvas Multidimensionales: Fijando la Escala Térmica $T_{\rm RH}$

En el análisis del régimen FIMP, variar simultáneamente $M_V$ y $T_{\text{RH}}$ dificulta identificar qué efecto proviene de la cinemática de producción del mediador y cuál del enfriamiento del plasma. 

Para aislar el impacto de la masa del mediador $M_V$:
1. Fijamos una escala térmica común (en este caso, un recalentamiento a baja escala: $T_{\text{RH}} = 0.01\text{ GeV}$).
2. Extraemos mediante `.get_group()` las secciones transversales correspondientes a $M_V \in \{200, 500, 800\}\text{ GeV}$.
3. Comparamos en un único panel la supresión de la densidad reliquia $\Omega_N h^2$ a medida que el mediador se vuelve más pesado.

---

### Implementación y Buenas Prácticas de Trazado

Cuando llamamos a `.get_group()` sobre cortes independientes, es crucial asegurar que cada subconjunto:
* Filtre el acoplamiento Yukawa unívocamente (`yX11 == 1e-13`).
* Se ordene monótonamente por masa (`sort_values('mN1')`).
* Descarte posibles colisiones numéricas de corridas paralelas (`drop_duplicates`).

---

# Problemas

A continuación se presentan 5 problemas diseñados para evaluar y afianzar las competencias de manipulación de datos con Pandas (`loc`, `iloc`, `np.where`, `groupby`, `.get_group()`, filtros vectorizados) y su posterior visualización física con Matplotlib a partir del dataset de simulaciones `data` (o `datafimp`).

---

### Problema 1: 

**Contexto Físico:**  
En modelos con mediadores vectoriales $V$ que decaen en pares de fermiones oscuros $N_1$ ($V \to N_1 N_1$), el canal de decaimiento en capa de masa (*on-shell*) está cinemáticamente permitido si y solo si la masa del mediador duplica al menos la masa del estado final:
$$M_V \ge 2 m_{N_1} \iff n \equiv \frac{M_V}{m_{N_1}} \ge 2$$
Si $n < 2$, el decaimiento decae a procesos de orden superior o dispersiones térmicas fuertemente suprimidas.

**Enunciado:**
1. Sobre el DataFrame completo `data`, cree una columna llamada `n` definida como el cociente $M_V / m_{N_1}$.
2. Utilizando **`np.where`**, cree una columna categórica llamada `CanalDecaimiento`:
   * Asigne el valor `'Permitido'` si $n \ge 2.0$.
   * Asigne el valor `'Bloqueado'` si $n < 2.0$.
3. Calcule mediante Pandas el porcentaje de puntos simulados que tienen el canal cinemáticamente bloqueado para cada régimen cosmológico (`FIMP` vs `WIMP`).

---

### Problema 2: 

**Contexto Físico:**  
La densidad reliquia de materia oscura observada por la colaboración Planck se sitúa en $\Omega_{\rm DM} h^2 \approx 0.12$. Consideramos una banda de viabilidad a $3\sigma$ dada por:
$$0.111 \le \Omega h^2 \le 0.129$$
Puntos con $\Omega h^2 < 0.111$ representan materia oscura sub-abundante (requieren componentes adicionales), mientras que $\Omega h^2 > 0.129$ sobrecierran el universo y quedan cosmológicamente descartados.

**Enunciado:**
1. Agregue una columna llamada `EstadoPlanck` inicializada con el valor por defecto `'Incompatible'`.
2. Utilice **`.loc` con máscaras booleanas** para clasificar el espacio de parámetros en tres etiquetas dentro de `EstadoPlanck`:
   * `'Sub-abundante'` si $\Omega h^2 < 0.111$.
   * `'Viable'` si $0.111 \le \Omega h^2 \le 0.129$.
   * `'Sobredensidad'` si $\Omega h^2 > 0.129$.
3. Filtre y muestre con `.loc` únicamente las columnas `['MV', 'mN1', 'TRH', 'yX11', 'Omega']` para las 5 filas con menor masa $m_{N_1}$ que resultaron ser `'Viable'` en el régimen FIMP.

---

### Problema 3: 

**Contexto Físico:**  
Para cada rebanada fija del plano de masas $(M_V, m_{N_1})$, queremos encontrar cuál fue el valor exacto del acoplamiento Yukawa $y_{X_{11}}$ y de la temperatura $T_{\rm RH}$ que produjo la densidad reliquia más cercana a la observación central $\Omega h^2 = 0.12$.

**Enunciado:**
1. En el DataFrame `datafimp`, cree una columna `delta_omega = (datafimp['Omega'] - 0.12).abs()`.
2. Utilice `groupby(['MV', 'mN1'])` junto con el método de reducción **`.idxmin()`** para extraer el índice de la fila que minimiza `delta_omega` para cada par $(M_V, m_{N_1})$.
3. Construya un nuevo DataFrame llamado `df_planck_optimo` indexando los datos con `.loc` a partir de los índices obtenidos.
4. Con `df_planck_optimo`, calcule la media y desviación estándar de $y_{X_{11}}$ agrupando únicamente por $M_V$.

---

### Problema 4: D

**Contexto Físico:**  
Deseamos comprobar la consistencia numérica de un bloque de escaneo específico antes de ejecutar análisis globales. Para una escala térmica intermedia $T_{\rm RH} = 1.0\text{ GeV}$ y un mediador $M_V = 500\text{ GeV}$, evaluaremos el salto en la densidad reliquia al cruzar el umbral cinemático.

**Enunciado:**
1. Agrupe `datafimp` por `['MV', 'TRH']` y utilice **`.get_group()`** para extraer en una sola línea el corte con $M_V = 500.0\text{ GeV}$ y $T_{\rm RH} = 1.0\text{ GeV}$.
2. Sobre el bloque extraído:
   * Filtre para un Yukawa fijo $y_{X_{11}} = 10^{-13}$.
   * Elimine duplicados sobre `mN1` con `.drop_duplicates()`.
   * Ordene monótonamente por masa con `.sort_values('mN1')`.
3. Grafique en escala log-log $\Omega_N h^2$ vs $m_{N_1}$.


---

### Problema 5: 

**Contexto Físico:**  
Se requiere construir una figura lista para publicación científica que compare el espacio de fases $(m_{N_1}, \Omega_N h^2)$ para tres valores del mediador:
$$M_V \in \{200, 500, 800\}\ \text{GeV}$$
fijando $y_{X_{11}} = 0.1$ y trazando las curvas continuas de evolución térmica para todas las temperaturas $T_{\rm RH}$ disponibles.

**Enunciado:**
1. Filtre el subconjunto de datos que contenga los tres valores de $M_V$ usando **`.isin()`**, con $y_{X_{11}} = 10^{-1}$, y limpie duplicados con `.drop_duplicates(subset=['MV', 'TRH', 'mN1'])`.
2. Cree una figura con 3 subplots horizontales compartiendo el eje Y (`sharey=True`).
3. Construya un diccionario de ejes `ax_map = dict(zip([200, 500, 800], axes))`.
4. Utilice un bucle `for (mv, trh), grupo in sub.groupby(['MV', 'TRH']):` para:
   * Ordenar cada grupo por `mN1`.
   * Trazar la curva en el eje correspondiente `ax_map[mv]`, asignando un color o etiqueta representativa según la temperatura $T_{\rm RH}$.
5. En cada panel, dibuje la banda sombreada de Planck ($0.111 \le \Omega h^2 \le 0.129$).